# Avancer le watermark d'ingestion

Associez `lh_meridian_hr` comme lakehouse par défaut. Dans Fabric, marquez la cellule 2 comme cellule de paramètres afin que le pipeline puisse remplacer les valeurs lors de l'avancement du watermark.

Ce notebook s'exécute comme **dernière** activité du pipeline et ne fait qu'avancer le watermark. La création et l'initialisation de `bronze.ingestion_watermark` sont gérées une seule fois, au début, par `nb_setup_lakehouse`.

## Paramètres (marquer cette cellule comme cellule de paramètres)

**Résumé.** Déclare les deux valeurs que le pipeline remplace lors de l'exécution : le pipeline auquel appartient le watermark et l'horodatage à enregistrer.

<details>
<summary>Détails ligne par ligne</summary>

- `pipeline_name = "workforce_events"` — la clé qui identifie la ligne de ce pipeline dans la table de watermark.
- `watermark_timestamp = "2020-12-01 00:00:00"` — la valeur à enregistrer; le pipeline transmet le `watermark` renvoyé par le notebook de découverte.

</details>

In [ ]:
pipeline_name = "workforce_events"
watermark_timestamp = "2020-12-01 00:00:00"

## Avancer le watermark

**Résumé.** Valide les paramètres, puis exécute un MERGE du nouvel horodatage dans l'unique ligne de ce pipeline dans `bronze.ingestion_watermark`.

<details>
<summary>Détails ligne par ligne</summary>

- `from datetime import datetime` — sert à analyser et valider l'horodatage fourni.
- Les contrôles `if ... raise ValueError` rejettent un `pipeline_name` vide et un horodatage qui ne correspond pas au premier jour d'un mois.
- `spark.createDataFrame([...])` + `createOrReplaceTempView("watermark_input")` — construisent une source à une ligne pour le MERGE.
- Le MERGE met à jour `watermark_timestamp`/`updated_at` lorsque la ligne existe; l'insertion **`WHEN NOT MATCHED`** sert de filet de sécurité si ce notebook est exécuté seul avant le notebook de configuration.
- Le `SELECT ... show(...)` final affiche les lignes de watermark actuelles pour confirmation.

</details>

In [ ]:
from datetime import datetime

if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_watermark = datetime.strptime(watermark_timestamp, "%Y-%m-%d %H:%M:%S")
if parsed_watermark.day != 1:
    raise ValueError("watermark_timestamp must be the first day of a month")

watermark_input = spark.createDataFrame(
    [(pipeline_name, parsed_watermark)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_input.createOrReplaceTempView("watermark_input")

spark.sql("""
MERGE INTO bronze.ingestion_watermark AS target
USING watermark_input AS source
ON target.pipeline_name = source.pipeline_name
WHEN MATCHED THEN UPDATE SET
    target.watermark_timestamp = source.watermark_timestamp,
    target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
    VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
""")

spark.sql("SELECT * FROM bronze.ingestion_watermark ORDER BY pipeline_name").show(truncate=False)